# TA-RandBox on Colab: quick start

Installs the environment, downloads RandBox's task-1 weights, checks the model end to end (a forward pass with and
without temporal proposals) and runs TA-RandBox on a short video.

Runtime: **GPU** (Runtime > Change runtime type). Evaluating on TAO needs the TAO frames (hundreds of GB); see the
last section.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())

## 1. Code and dependencies

Detectron2 is built from source against Colab's PyTorch (a few minutes).

In [ ]:
REPO_URL = "https://github.com/elTxolas/RandomBoxesObjectDetector.git"  # update if the repository moves
BRANCH = "master"
!git clone -q --branch $BRANCH $REPO_URL /content/TA-RandBox || (cd /content/TA-RandBox && git pull -q)
%cd /content/TA-RandBox
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q -r requirements.txt gdown

In [ ]:
import detectron2, randbox
print("detectron2", detectron2.__version__)

## 2. Weights

RandBox's released task-1 model (20 known classes), from the [RandBox README](https://github.com/Yanghao-Wang/RandBox#models).

In [ ]:
import os
os.makedirs("models", exist_ok=True)
if not os.path.exists("models/randbox_t1.pth"):
    !gdown -q 1HjvHm7YQ9VMUbU5mDIGmXg8BWtDqmGGn -O models/randbox_t1.pth
!ls -lh models/

## 3. End-to-end check

Builds the model for each inference method and runs two consecutive frames: the second one receives the first
one's tracked boxes as temporal proposals.

In [ ]:
import sys, time, numpy as np, torch
sys.path.insert(0, "/content/TA-RandBox")
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from randbox import add_RandBox_config
from randbox.util.model_ema import add_model_ema_configs
from tools.eval_tao import METHODS

def load(method):
    cfg = get_cfg(); add_RandBox_config(cfg); add_model_ema_configs(cfg)
    cfg.merge_from_file("configs/t1.yaml"); cfg.merge_from_list(METHODS[method])
    cfg.MODEL.WEIGHTS = "models/randbox_t1.pth"; cfg.freeze()
    model = build_model(cfg); DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS); model.eval()
    return model

import cv2
from detectron2.data import transforms as T
VIDEO_URL = "https://github.com/opencv/opencv/raw/4.x/samples/data/vtest.avi"  # OpenCV's pedestrian sample
!wget -q -nc -O /content/input.avi "$VIDEO_URL"
ok, bgr = cv2.VideoCapture("/content/input.avi").read()
assert ok, "could not read the sample video"
h, w = bgr.shape[:2]
image = T.ResizeShortestEdge([800, 800], 1333).get_transform(bgr).apply_image(bgr)
inputs = [{"image": torch.as_tensor(image.astype("float32").transpose(2, 0, 1)), "height": h, "width": w}]
for method in ["randbox-500", "ta-randbox", "randbox-10k"]:
    model = load(method)
    with torch.no_grad():
        out1, tracked = model(inputs, return_tracked=True)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        out2, _ = model(inputs, prev_boxes=tracked, return_tracked=True)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
    print(f"{method:12s} detections {len(out2[0]['instances']):4d} | tracked {0 if tracked is None else len(tracked):3d} | {dt:.3f} s/frame")
    del model; torch.cuda.empty_cache()

## 4. TA-RandBox on a video

Any video works (`--input`); here, the sample downloaded above.

In [ ]:
!python tools/demo_video.py --input /content/input.avi --output /content/ta_randbox.mp4 --method ta-randbox --fps 10 MODEL.WEIGHTS models/randbox_t1.pth

In [ ]:
!ffmpeg -y -loglevel error -i /content/ta_randbox.mp4 -vcodec libx264 /content/ta_randbox_h264.mp4
from IPython.display import Video
Video("/content/ta_randbox_h264.mp4", embed=True, width=720)

## 5. Evaluation on TAO

Download TAO's frames and annotations from [taodataset.org](https://taodataset.org) into `datasets/tao/`
(`frames/train/<source>/<video>/`, `annotations/train.json`); mounting Google Drive is the practical way on Colab.
Then (about 2-3 h per method on LaSOT with one V100-class GPU; `--max-videos 5` for a quick run):

In [ ]:
# !python tools/prepare_tao.py --tao-root datasets/tao --subsets LaSOT --out datasets/tao/frames_lasot.pkl
# !python tools/eval_tao.py --config-file configs/t1.yaml --frames datasets/tao/frames_lasot.pkl --method ta-randbox --max-videos 5 MODEL.WEIGHTS models/randbox_t1.pth OUTPUT_DIR output/lasot_ta
# !bash scripts/eval_tao.sh LaSOT models/randbox_t1.pth